# CryptoTrace - Advanced GNN Training (GATv2)
This notebook trains an advanced Graph Attention Network v2 (GATv2) on the Elliptic Bitcoin Dataset.
It includes class weighting, temporal splits to avoid data leakage, and learning rate scheduling to maximize accuracy and F1 score for detecting illicit money laundering nodes.

**Step 1:** Go to `Runtime > Change runtime type` and select `T4 GPU`.

In [ ]:
!pip install torch-geometric

In [ ]:
import os
import torch
import torch.nn.functional as F
from torch_geometric.datasets import EllipticBitcoinDataset
from torch_geometric.nn import GATv2Conv
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

print("Loading EllipticBitcoinDataset...")
dataset = EllipticBitcoinDataset(root='./data/Elliptic')
data = dataset[0]

class AdvancedGNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4):
        super(AdvancedGNN, self).__init__()
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads, dropout=0.4)
        self.conv2 = GATv2Conv(hidden_channels * heads, hidden_channels, heads=heads, dropout=0.4)
        self.conv3 = GATv2Conv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.4)

    def forward(self, x, edge_index):
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.4, training=self.training)
        x2 = F.elu(self.conv2(x, edge_index))
        x = x + x2 
        x = F.dropout(x, p=0.4, training=self.training)
        x = self.conv3(x, edge_index)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

data = data.to(device)

model = AdvancedGNN(
    in_channels=dataset.num_node_features,
    hidden_channels=64,
    out_channels=2,
    heads=4
).to(device)

optimizer = Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, verbose=True)

labeled_mask = (data.y == 0) | (data.y == 1)
train_mask = data.train_mask & labeled_mask
test_mask = data.test_mask & labeled_mask

num_illicit = (data.y[train_mask] == 0).sum().item()
num_licit = (data.y[train_mask] == 1).sum().item()
total_train = num_illicit + num_licit

weight_illicit = total_train / (2 * num_illicit) if num_illicit > 0 else 1.0
weight_licit = total_train / (2 * num_licit) if num_licit > 0 else 1.0
class_weights = torch.tensor([weight_illicit, weight_licit], dtype=torch.float).to(device)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[train_mask], data.y[train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def test():
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        loss_val = criterion(out[test_mask], data.y[test_mask]).item()
        
        true_positive = ((pred[test_mask] == 0) & (data.y[test_mask] == 0)).sum().item()
        false_positive = ((pred[test_mask] == 0) & (data.y[test_mask] == 1)).sum().item()
        false_negative = ((pred[test_mask] == 1) & (data.y[test_mask] == 0)).sum().item()
        
        accuracy = (pred[test_mask] == data.y[test_mask]).sum().item() / test_mask.sum().item()
        precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
        recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
    return loss_val, accuracy, precision, recall, f1

epochs = 500
best_f1 = 0
patience_counter = 0
early_stop_patience = 50
model_path = 'model_advanced.pth'

for epoch in range(1, epochs + 1):
    train_loss = train()
    val_loss, acc, prec, rec, f1 = test()
    scheduler.step(val_loss)
    
    if f1 > best_f1:
        best_f1 = f1
        patience_counter = 0
        torch.save(model.state_dict(), model_path)
    else:
        patience_counter += 1
        
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Acc: {acc:.4f} | Illicit F1: {f1:.4f} | Recall: {rec:.4f}")
              
    if patience_counter >= early_stop_patience:
        print(f"Early stopping triggered at epoch {epoch}. Best Illicit F1: {best_f1:.4f}")
        break

print(f"Training complete. Best model weights saved to {model_path}.")

In [ ]:
from google.colab import files
files.download('model_advanced.pth')